# 06 — Attribution, ONNX, real-time scoring, and the Flask demo

**Demonstrates:** the person + gender attribution pipeline and its fail-safe invariant, the
ONNX export's parity check against the Keras model, measured CPU latency and sustainable FPS,
a live sliding-window score over a real raw video through the exact shipped artefact, and the
Flask app serving that same artefact end to end.

**Maps to:** report §2.7/§5 (SUSAN-style general violence detector + person detection + gender
module), §1 (ONNX export), §1/§2.8 (real-time CCTV input to alert), `docs/SPRINT_PLAN.md`
Sprints 4 and 5.


In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import json  # noqa: E402

import pandas as pd  # noqa: E402

manifest = pd.read_parquet("data/manifests/clips.parquet")


## Attribution pipeline, live, on a real AIRTLab clip

Person detection is YOLOv8n + ByteTrack; gender attribution is zero-shot CLIP scoring each tracked person's best crop against two text prompts. Both run live here against a real clip, not a stub.

In [2]:
import time

from safestreets.attribution.gender import GenderClassifierConfig, ZeroShotGenderClassifier
from safestreets.attribution.person import PersonDetector, PersonDetectorConfig, summarize_tracks
from safestreets.attribution.pipeline import AttributionPipelineConfig, build_alert
from safestreets.evaluation.evaluate import load_checkpoint, predict_scores
from safestreets.training.train import load_feature_split

attr_raw = __import__("yaml").safe_load(Path("configs/attribution.yaml").read_text())
person_cfg = PersonDetectorConfig(**attr_raw["person"])
gender_cfg = GenderClassifierConfig(**attr_raw["gender"])
pipeline_cfg = AttributionPipelineConfig(
    violence_threshold=attr_raw["violence_threshold"], priority_gender=attr_raw["priority_gender"]
)

person_detector = PersonDetector(person_cfg)
gender_classifier = ZeroShotGenderClassifier(gender_cfg)

sample = manifest[(manifest.dataset == "airtlab") & (manifest.split == "test")].iloc[0]
print(f"clip: {sample.clip_id}  label={sample.label}  path={sample.path}")


/opt/anaconda3/envs/safestreets/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/10 17:08:48 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /opt/anaconda3/envs/safestreets/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


clip: airtlab_violent_cam1_106  label=1  path=data/raw/airtlab/violence-detection-dataset/violent/cam1/106.mp4


In [3]:
model = load_checkpoint("artifacts/checkpoints/finetuned_airtlab_lstm_head.keras")
x_test, y_test, clip_ids = load_feature_split("data/features", ["airtlab"], "test")
idx = clip_ids.index(sample.clip_id)
p_violence = float(predict_scores(model, x_test[idx : idx + 1])[0])
print(f"p_violence (finetuned_airtlab checkpoint) = {p_violence:.4f}")


2026-10-10 17:08:56.281305: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-10-10 17:08:56.281339: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-10-10 17:08:56.281344: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-10-10 17:08:56.281366: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-10 17:08:56.281381: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


2026-10-10 17:09:01.666680: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


p_violence (finetuned_airtlab checkpoint) = 0.9923


In [4]:
t0 = time.time()
detections = person_detector.track_video(sample.path) if person_detector.available() else None
track_summary = summarize_tracks(detections) if detections else None

crops = []
if detections:
    best = {}
    import cv2

    for det in detections:
        cur = best.get(det["track_id"])
        if cur is None or det["conf"] > cur["conf"]:
            best[det["track_id"]] = det
    cap = cv2.VideoCapture(sample.path)
    for det in best.values():
        cap.set(cv2.CAP_PROP_POS_FRAMES, det["frame"] * person_cfg.every_n_frames)
        ok, frame = cap.read()
        if not ok:
            continue
        x1, y1, x2, y2 = (int(v) for v in det["bbox"])
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, w), min(y2, h)
        if x2 > x1 and y2 > y1:
            crops.append(cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    cap.release()

gender_available = gender_classifier.available() and crops
gender_results = gender_classifier.classify(crops) if gender_available else None
alert = build_alert(
    p_violence, pipeline_cfg.violence_threshold, pipeline_cfg.priority_gender,
    detections, gender_results,
)
print(f"attribution pipeline wall time: {time.time() - t0:.1f}s")
print(json.dumps({**alert, "track_summary": track_summary}, indent=2))


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



attribution pipeline wall time: 13.3s
{
  "alert": true,
  "p_violence": 0.9923330545425415,
  "threshold": 0.2,
  "attribution_available": true,
  "n_people": 3,
  "genders_detected": [
    "woman",
    "woman",
    "woman"
  ],
  "priority": "elevated",
  "track_summary": {
    "n_detections": 94,
    "n_tracks": 3,
    "frames_per_track": {
      "1": 37,
      "2": 33,
      "4": 24
    }
  }
}


## The fail-safe invariant, demonstrated

`build_alert` computes `alert` from `p_violence` and `threshold` alone, before it even reads `person_detections`/`gender_results` (`safestreets/attribution/pipeline.py`). A wrong or missing gender label can change `priority`, never whether the alert fires. Proven here directly: force the gender module to report no women detected, and confirm the alert still fires on a high violence score.

In [5]:
forced_no_women = [{"label": "man", "score": 0.9, "probs": {"man": 0.9, "woman": 0.1}}]
alert_forced = build_alert(0.99, threshold=0.2, priority_gender="woman",
                            person_detections=[{"track_id": 0}], gender_results=forced_no_women)
assert alert_forced["alert"] is True
assert alert_forced["priority"] == "standard"
print("p_violence=0.99 with gender forced to all-man still raises the alert "
      f"(alert={alert_forced['alert']}, priority={alert_forced['priority']!r}); "
      "attribution changed priority, never the alert itself")


p_violence=0.99 with gender forced to all-man still raises the alert (alert=True, priority='standard'); attribution changed priority, never the alert itself


## Attribution across the AIRTLab test set

Qualitative results from the committed Sprint 4 run (`scripts/attribution_eval.py`, 48 clips) — see `docs/ETHICS.md` for why this is reported as a qualitative check, not a measured accuracy number (no person-level gender ground truth exists in this pipeline).

In [6]:
attr_eval = json.loads(Path("artifacts/reports/attribution_eval.json").read_text())
genders_flat = [g for r in attr_eval["results"] for g in r["alert"]["genders_detected"]]
print(f"n_clips={attr_eval['n_clips']}  "
      f"person_detector_available={attr_eval['person_detector_available']}  "
      f"gender_classifier_available={attr_eval['gender_classifier_available']}")
print(f"gender labels predicted across all detected people: "
      f"woman={genders_flat.count('woman')}  man={genders_flat.count('man')}")


n_clips=48  person_detector_available=True  gender_classifier_available=True
gender labels predicted across all detected people: woman=154  man=73


## ONNX export: parity and latency

`scripts/export_onnx.py` bundles MobileNetV2 feature extraction and the fine-tuned head into one ONNX graph (ADR-006). Real numbers from the committed export run:

In [7]:
parity = json.loads(Path("artifacts/reports/onnx_parity.json").read_text())
latency = json.loads(Path("artifacts/reports/latency.json").read_text())
parity_status = "1e-4 passed" if parity["parity_passed"] else "FAILED"
print(f"parity: n={parity['n_clips']} max_abs_diff={parity['max_abs_diff']:.3e} "
      f"(tol {parity_status})")
print(f"roc_auc keras={parity['roc_auc_keras']:.6f} onnx={parity['roc_auc_onnx']:.6f}")
print(f"latency: p50={latency['p50_ms']:.2f}ms  p95={latency['p95_ms']:.2f}ms  "
      f"p99={latency['p99_ms']:.2f}ms")
print(f"sustainable_fps={latency['sustainable_fps']:.1f} (stride {latency['stride']}), "
      f"realtime_headroom={latency['realtime_headroom']:.1f}x")


parity: n=325 max_abs_diff=5.244e-05 (tol 1e-4 passed)
roc_auc keras=0.969105 onnx=0.969105
latency: p50=111.08ms  p95=134.41ms  p99=167.90ms
sustainable_fps=59.5 (stride 8), realtime_headroom=18.6x


The parity note: AIRTLab test clips were part of the ADR-005 fine-tune pool, so this AUC checks
runtime agreement between the two inference engines on the same model, not generalisation.

## Live sliding-window score on a real raw video

`InferenceEngine` (onnxruntime only, no TensorFlow) scores the whole clip and replays it through the exact streaming scorer (EMA smoothing, hysteresis) the Flask app and a live camera feed would use.

In [8]:
from safestreets.inference.engine import InferenceEngine

engine = InferenceEngine()
print(f"model_version={engine.model_version}  git_sha={engine.git_sha[:12]}  "
      f"threshold={engine.threshold}")

airtlab_test_violent = (
    (manifest.dataset == "airtlab") & (manifest.split == "test") & (manifest.label == 1)
)
violent_sample = manifest[airtlab_test_violent].iloc[0]
t0 = time.time()
result = engine.analyse_video(violent_sample.path)
print(f"scored {violent_sample.clip_id} in {time.time() - t0:.2f}s")
print(f"clip_score={result['clip_score']:.4f}  verdict={result['verdict']!r}  "
      f"n_windows={len(result['windows'])}  alert_events={result['alert_events']}")
pd.DataFrame(result["windows"])


model_version=finetuned_airtlab-73ebf22e032d  git_sha=d8b8335c2c63  threshold=0.2


scored airtlab_violent_cam1_106 in 3.20s
clip_score=0.9923  verdict='violent'  n_windows=1  alert_events=1


,index,start_frame,end_frame,raw_score,smoothed_score,alert_active,alert_started,start_s,end_s
0,0,0,135,0.985604,0.985604,True,True,0.0,4.5


## The Flask app, end to end

Same `InferenceEngine`, served through the real Flask app (`safestreets.web.create_app`) via its test client, exercising the actual upload route rather than calling the engine directly. Rendered inline below in place of a screenshot.

In [9]:
from IPython.display import HTML

from safestreets.web import create_app

app = create_app()
client = app.test_client()

index_resp = client.get("/")
print(f"GET / -> {index_resp.status_code}")

with open(violent_sample.path, "rb") as f:
    analyse_resp = client.post(
        "/analyse", data={"video": (f, "clip.mp4")}, content_type="multipart/form-data"
    )
print(f"POST /analyse -> {analyse_resp.status_code}")
HTML(analyse_resp.get_data(as_text=True))


GET / -> 200


POST /analyse -> 200


## Limitations

- **The gender module's accuracy is unmeasured** (zero-shot CLIP, no labelled ground truth in
  this pipeline); treat the label distribution above as qualitative, never as an accuracy claim
  (`docs/ETHICS.md`).
- **Latency and parity were measured on this machine, once**, under its own background load at
  measurement time; they are not a guarantee across hardware.
- **The Flask demo above runs through Flask's test client, not a live HTTP server or browser**,
  so no network stack or JavaScript is exercised; `safestreets/web/routes.py`'s `/analyse` form
  and `/api/analyse` JSON endpoint are otherwise identical code paths to what a browser would hit.
